# 61 — RNA Pseudobulk DESeq2 (Cross-Species DE)

Runs after `scripts/create_rna_pseudobulk.py`.
Uses the same DESeq2 methodology as the ATAC pipeline (Section 11 of deseq2_utils.R).

In [ ]:
# Cell 1: Packages & utilities
suppressPackageStartupMessages({
  library(DESeq2); library(arrow)
  library(ggplot2); library(dplyr); library(tibble); library(readr)
})
source("../src/deseq2_utils.R")
message("Packages loaded.")

In [ ]:
# Cell 2: Config
BASE_RNA <- "/links/groups/treutlein/USERS/jjans/analysis/adult_intestine/rna"
PB_DIR   <- file.path(BASE_RNA, "pseudobulk_deseq2")
OUT_DIR  <- file.path(BASE_RNA, "pseudobulk_deseq2")   # results saved here too
SPECIES  <- c("Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset")

# QC thresholds (RNA is much deeper than ATAC regions; use counts not cells)
MIN_COUNTS <- 500
MIN_CELLS  <- 10

# Check pseudobulk files exist
for (sp in SPECIES) {
  f <- file.path(PB_DIR, sp, "pseudobulk_counts.parquet")
  if (!file.exists(f)) stop("Missing pseudobulk for ", sp, ": run create_rna_pseudobulk.py first")
}
message("All pseudobulk files found.")

In [ ]:
# Cell 3: Load & merge RNA pseudobulk data
raw_rna <- load_rna_pseudobulk(PB_DIR, SPECIES)

# Inner join: shared genes across all species
merged_rna <- merge_rna_pseudobulk(raw_rna$all_counts, raw_rna$all_meta, join_type = "inner")
counts_rna <- merged_rna$counts
meta_rna   <- merged_rna$meta

message("Shared genes: ", nrow(counts_rna))
message("Total samples: ", ncol(counts_rna))
print(table(meta_rna$species, meta_rna$cell_type))

In [ ]:
# Cell 4: Aggregate across regions (Individual x cell_type, summing Duodenum + Colon)
agg <- aggregate_rna_pseudobulk(counts_rna, meta_rna)
counts_agg <- agg$counts
meta_agg   <- agg$meta

message("After region aggregation: ", ncol(counts_agg), " samples")
message("Samples per species:")
print(table(meta_agg$species))
print(table(meta_agg$cell_type))

In [ ]:
# Cell 5: Sample QC — remove low-coverage pseudobulks
meta_agg$total_counts <- colSums(counts_agg)
keep <- meta_agg$total_counts >= MIN_COUNTS & meta_agg$n_cells >= MIN_CELLS
message("Samples passing QC (counts>=", MIN_COUNTS, ", cells>=", MIN_CELLS, "): ",
        sum(keep), "/", ncol(counts_agg))
counts_agg <- counts_agg[, keep, drop = FALSE]
meta_agg   <- meta_agg[keep, , drop = FALSE]

# Factor setup
meta_agg$species   <- factor(meta_agg$species, levels = SPECIES)
meta_agg$cell_type <- factor(make.names(as.character(meta_agg$cell_type)))
meta_agg$donor     <- factor(meta_agg$donor)

# Save QC-filtered pseudobulk RDS for downstream notebooks
saveRDS(list(counts = counts_agg, meta = meta_agg),
        file.path(PB_DIR, "rna_pb_data_aggregated.rds"))
message("Saved rna_pb_data_aggregated.rds")

In [ ]:
# Cell 6: PCA sanity check
vst_rna <- vst(DESeqDataSetFromMatrix(
  countData = counts_agg,
  colData   = meta_agg,
  design    = ~ species
), blind = TRUE)

pca_df <- as.data.frame(prcomp(t(assay(vst_rna)), scale. = FALSE)$x[, 1:3])
pca_df <- cbind(pca_df, meta_agg)

p <- ggplot(pca_df, aes(PC1, PC2, color = species, shape = cell_type)) +
  geom_point(size = 2, alpha = 0.8) +
  theme_bw() + ggtitle("RNA pseudobulk PCA (VST, shared genes)")
ggsave(file.path(OUT_DIR, "_summary", "RNA_PCA_species.pdf"), p, width = 9, height = 6)
message("PCA plot saved.")

In [ ]:
# Cell 7: DESeq2 — Species-specific (each species vs rest, like shared-peak ATAC)
message("\n========== Species-specific RNA DE ==========")
rna_sp_res <- run_deseq2_rna_species(
  counts_rna   = counts_agg,
  meta_rna     = meta_agg,
  species      = SPECIES,
  out_dir      = OUT_DIR,
  min_samples  = 2,
  filter_outliers = TRUE
)
message("Species-specific RNA DE done. Saved to rna_differential_results/species_specific/")

In [ ]:
# Cell 8: DESeq2 — Evolutionary contrasts (same 19 contrasts as ATAC)
message("\n========== Evolutionary RNA DE ==========")
evo_contrasts <- define_evolutionary_contrasts()
message("Running ", length(evo_contrasts), " contrasts")

rna_evo_res <- run_deseq2_rna_evolutionary(
  counts_rna      = counts_agg,
  meta_rna        = meta_agg,
  evo_contrasts   = evo_contrasts,
  out_dir         = OUT_DIR,
  min_samples     = 2,
  filter_outliers = TRUE
)
message("Evolutionary RNA DE done. Saved RNA_DE_res_list.rds")

In [ ]:
# Cell 9: Quick summary — how many DE genes per cell type × contrast?
summary_rows <- list()
for (ct in names(rna_evo_res)) {
  for (cn in names(rna_evo_res[[ct]])) {
    res <- as.data.frame(rna_evo_res[[ct]][[cn]])
    n_up   <- sum(!is.na(res$padj) & res$padj < 0.05 & res$log2FoldChange > 1)
    n_down <- sum(!is.na(res$padj) & res$padj < 0.05 & res$log2FoldChange < -1)
    summary_rows[[paste(ct, cn)]] <- data.frame(
      cell_type = ct, contrast = cn, n_up = n_up, n_down = n_down
    )
  }
}
de_summary <- do.call(rbind, summary_rows)
write.csv(de_summary, file.path(OUT_DIR, "_summary", "RNA_DE_summary.csv"),
          row.names = FALSE)
message("\nDE gene counts per cell type × contrast:")
print(de_summary[order(de_summary$n_up, decreasing = TRUE), ][1:30,])

In [ ]:
# Cell 10: Final checkpoint
message("\n=== RNA DE notebook complete ===")
message("Output: ", file.path(OUT_DIR, "_summary/"))
message("  RNA_DE_res_list.rds              (evolutionary contrasts)")
message("  RNA_DE_species_res_list.rds")
message("  RNA_DE_summary.csv")
message("Per-CT results: ", OUT_DIR, "/{CellType}/rna_evolutionary/ and /rna_species/")